**Name**: Son Nguyen

**Website Link**: https://hpsonnguyen.github.io/LoL-analysis/

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc80_utils import *

In [2]:
%load_ext autoreload
%autoreload 2

## Step 1: Introduction

**Problem Statement**: Identify the key factors contributing to a team's success in winning a League of Legends esports game, and develop a predictive model to forecast game outcomes based on these factors.

**Questions**:
- What are the most significant factors influencing a team's chances of winning a League of Legends game?
- How can we quantify and weight these factors to predict game outcomes?

## Step 2: Data Cleaning and Exploratory Data Analysis
### Read and Clean Dataset

In [3]:
# List of columns that have mix types and need to be read as strings
columns_to_convert = [
    'url', 'split', 'playername', 'playerid', 'teamname', 'teamid',
    'champion', 'ban1', 'ban2', 'ban3', 'ban4', 'ban5',
    'pick1', 'pick2', 'pick3', 'pick4', 'pick5'
]

# Read the CSV file, specifying dtype for the problematic columns
raw = pd.read_csv(
    'data/2022_match.csv',
    dtype={col: 'str' for col in columns_to_convert},
    na_values=['', 'nan', 'None']
)

# Convert binary nominal columns (contain only 0's and 1's) to True/False, preserve NaN
bin_cols = list(raw.columns[raw.columns.str.contains('first')]) + ['result', 'playoffs']
raw[bin_cols] = raw[bin_cols].where(raw[bin_cols].isna(), raw[bin_cols].astype(bool))

print('Shape:', raw.shape)
raw.head()

Shape: (148992, 131)


,gameid,datacompleteness,url,league,...,deathsat15,opp_killsat15,opp_assistsat15,opp_deathsat15
0,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,0.0,0.0,1.0,0.0
1,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,2.0,0.0,5.0,1.0
2,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,0.0,3.0,3.0,2.0
3,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,2.0,3.0,3.0,0.0
4,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,2.0,0.0,6.0,2.0


In [4]:
# Explore column names
print(raw.columns.to_list())

['gameid', 'datacompleteness', 'url', 'league', 'year', 'split', 'playoffs', 'date', 'game', 'patch', 'participantid', 'side', 'position', 'playername', 'playerid', 'teamname', 'teamid', 'champion', 'ban1', 'ban2', 'ban3', 'ban4', 'ban5', 'pick1', 'pick2', 'pick3', 'pick4', 'pick5', 'gamelength', 'result', 'kills', 'deaths', 'assists', 'teamkills', 'teamdeaths', 'doublekills', 'triplekills', 'quadrakills', 'pentakills', 'firstblood', 'firstbloodkill', 'firstbloodassist', 'firstbloodvictim', 'team kpm', 'ckpm', 'firstdragon', 'dragons', 'opp_dragons', 'elementaldrakes', 'opp_elementaldrakes', 'infernals', 'mountains', 'clouds', 'oceans', 'chemtechs', 'hextechs', 'dragons (type unknown)', 'elders', 'opp_elders', 'firstherald', 'heralds', 'opp_heralds', 'void_grubs', 'opp_void_grubs', 'firstbaron', 'barons', 'opp_barons', 'firsttower', 'towers', 'opp_towers', 'firstmidtower', 'firsttothreetowers', 'turretplates', 'opp_turretplates', 'inhibitors', 'opp_inhibitors', 'damagetochampions', 'dp

Upon exploring the data, we noticed that there are 2 instances where no team won a game. Since our primary question concerns the win/loss outcome of a game, we consider these data points to be outliers and they should be removed from the analysis dataset.

In [5]:
# Number of defeating is not equal to the number of winning (but they should be equal)
print(
    'The difference in numbers of defeating teams and winning teams:',
    (raw[~raw['result']].shape[0] - raw[raw['result']].shape[0]) // 12 # Each game has 12 rows
)

The difference in numbers of defeating teams and winning teams: 2


In [6]:
# These outlier games are
defect_result = raw.groupby('gameid')['result'].sum()
defect_games = defect_result[defect_result != 6].index
print('Remove these games:', defect_games.to_list())
raw = raw[~raw['gameid'].isin(defect_games)]

Remove these games: ['ESPORTSTMNT03_2788015', 'ESPORTSTMNT04_2170436']


In [7]:
"""
Functions to get the data of interest in later sections
"""

def get_game_rows(raw):
    """
    Takes in the raw DataFrame and returns a DataFrame
    containing 2 rows for every game, each row contains
    the data of Blue or Red team in that game
    """
    if 'position' in raw.columns:
        game = raw[raw['position'] == 'team']
    else:
        game = raw[raw['playername'].isna()]
    return game

def get_player_rows(raw):
    """
    Takes in the raw DataFrame and returns a DataFrame
    containing 10 rows for every game, each row contains
    the data of a player in that game
    """
    player = raw[~raw['playername'].isna()]
    return player

def get_tier_one(df):
    """
    Returns a DataFrame of only games within tier-one leagues,
    each game has 12 rows (10 player-rows 2 game-rows)
    """
    return df[df['league']
              .isin(['LCK', 'LPL',
                     'LEC', 'LCS',
                     'PCS', 'VCS',
                     'CBLOL', 'LLA'])]

def plot_wl_hist(game_df, col):
    """
    Shows histograms of win teams' and defeat teams'
    measurement in a column game_df
    """
    win_df = game_df[game_df['result']].drop(['result'], axis=1)
    defeat_df = game_df[~game_df['result']].drop(['result'], axis=1)

    def win_defeat_distr(win_df, defeat_df, col):
        df = (win_df[['gameid', col]]
            .merge(defeat_df[['gameid', col]], on='gameid')
            .set_index('gameid'))
        df.columns = [f'{col}_win', f'{col}_defeat']
        return df
    
    fig = px.histogram(
        win_defeat_distr(win_df, defeat_df, col),
        x=[f'{col}_win', f'{col}_defeat'],
        histnorm='probability',
        nbins=100,
        opacity=0.7,
        color_discrete_sequence=['#3288bd', '#d53e4f'],
        barmode='group',
        title=f'Distribution of {col} for Winning and Losing Teams',
    )
    
    fig.update_traces(
        name='Win',
        legendgroup='Win',
        selector=dict(name=f'{col}_win')
    )
    fig.update_traces(
        name='Lose',
        legendgroup='Lose',
        selector=dict(name=f'{col}_defeat')
    )
    fig.update_layout(
        legend_title="Team",
        xaxis_title=col,
        yaxis_title="Probability"
    )

    fig.show()

def get_position_stats(raw, position, stats=None):
    """
    Returns a DataFrame containing the rows of a specified position
    and columns specified as a list passed to `stats`
    """
    if stats is None:
        stats = raw.columns.drop(['gameid', 'result'])
        df = raw[raw['position'] == position]
    else:
        df = raw[raw['position'] == position][['gameid', 'result'] + stats]
    rename = {stat: f'{position}_{stat}' for stat in stats}
    return df.rename(columns=rename)

### Explore team data
#### Univariate analysis on team's statistics

To perform analysis on each team's overall statistics, we need to retrieve the rows that represent each team's data from the entire dataset. We will store this retrieved DataFrame in a variable called `game`.

In [8]:
game = get_game_rows(raw)
game.head()

,gameid,datacompleteness,url,league,...,deathsat15,opp_killsat15,opp_assistsat15,opp_deathsat15
10,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,6.0,6.0,18.0,5.0
11,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,5.0,5.0,10.0,6.0
22,ESPORTSTMNT01_2690219,complete,NaN,LCKC,...,3.0,3.0,3.0,1.0
23,ESPORTSTMNT01_2690219,complete,NaN,LCKC,...,1.0,1.0,1.0,3.0
34,8401-8401_game_1,partial,https://lpl.qq.com/es/stats.shtml?bmid=8401,LPL,...,NaN,NaN,NaN,NaN


Let's analyze the distributions of key game metrics such as kills, damage, and gold. Since each game involves two teams, we'll focus on comparing these metrics between the **winning** and **losing** teams to understand their impact on game outcomes.

To begin, we'll extract the relevant data from the `game` DataFrame.

In [9]:
crit_meas = game[['gameid', 'side', 'result', 'kills', 'deaths', 'assists',
                 'damagetochampions', 'dpm', 'damagemitigatedperminute',
                 'visionscore', 'totalgold', 'minionkills']]
# Missing values:
# 'damagetochampions': 2, 'dpm': 2, 'damagemitigatedperminute': 3638, 'visionscore': 2, 'minionkills': 3638
crit_meas = crit_meas.rename(columns={
    'kills': 'Kills',
    'deaths': 'Deaths',
    'assists': 'Assists',
    'damagetochampions': 'Damage to Champions',
    'dpm': 'Damage per Minute',
    'damagemitigatedperminute': 'Damage Mitigated per Minute',
    'visionscore': 'Vision Score',
    'totalgold': 'Total Gold',
    'minionkills': 'Minion Kills',
})
crit_meas.head()

,gameid,side,result,Kills,...,Damage Mitigated per Minute,Vision Score,Total Gold,Minion Kills
10,ESPORTSTMNT01_2690210,Blue,False,9,...,2364.73,197.0,47070,680.0
11,ESPORTSTMNT01_2690210,Red,True,19,...,2872.33,205.0,52617,792.0
22,ESPORTSTMNT01_2690219,Blue,False,3,...,3109.61,277.0,57629,994.0
23,ESPORTSTMNT01_2690219,Red,True,16,...,2868.42,346.0,71004,1013.0
34,8401-8401_game_1,Blue,True,13,...,NaN,162.0,45468,NaN


In [10]:
# Columns to plot
columns_to_plot = crit_meas.columns.drop(['gameid', 'side', 'result'])

# Populate subplot with histograms
for col in columns_to_plot:
    plot_wl_hist(crit_meas, col)

#### An aggregates statistic: Team's first baron analysis

To explore more patterns between winning and losing teams, we might examine the number of Barons a team killed and whether a team secured the first Baron. Statistics related to Baron kills may provide valuable insights into winning chances, as securing Baron grants a strategic advantage by granting the team that obtains the Baron buff an upper hand in destroying the opposing team's Nexus, the final objective to win the game. Moreover, Baron is typically targeted in the mid and late game stages, as it requires significant damage and team collaboration to defeat. Thus, having Barons signifies a team's control of the game as it nears its conclusion. Therefore, our analysis of Baron statistics may benefit later modeling efforts. With that said, let's analyze the association between the team that secured the first Baron and the game result.

In [11]:
baron_df = get_game_rows(raw)[['gameid', 'result', 'firstbaron']].dropna()
baron_df['firstbaron_True'] = baron_df['firstbaron'] == True
baron_df['firstbaron_False'] = baron_df['firstbaron'] == False
first_baron = baron_df.groupby('result')[['firstbaron_True', 'firstbaron_False']].mean()
first_baron

,firstbaron_True,firstbaron_False
result,,
False,0.14,0.86
True,0.81,0.19


In [12]:
# Create a heatmap for first_baron
trace = go.Heatmap(
    z=first_baron.values,
    x=first_baron.columns,
    y=first_baron.index,
    colorscale=[[0.0, '#d53e4f'], [0.5, '#ffffff'], [1.0, '#3288bd']],
    colorbar=dict(
        tickvals=[1, 0.5, 0],
        ticktext=['0.0', '0.5', '1.0'],
    ),
    text=[[f'Proportion: {value:.2f}' for value in row] for row in first_baron.values],
    hoverinfo='text'
)
layout = go.Layout(
    title='Relational Proportion between First Baron and Result',
    xaxis=dict(title='First Baron', tickvals=[0, 1], ticktext=['Yes', 'No']),
    yaxis=dict(title='Result', tickvals=[0, 1], ticktext=['Defeat', 'Win'])
)
fig = go.Figure(data=[trace], layout=layout)
fig.show()

The relationship between securing the first Baron and winning the game appears to be strongly polarized. This observation offers valuable insights for further exploration and modeling.

Upon further correlation exploration among the columns in the `game` dataset, we found no particularly interesting correlations between the numeric features. We mainly observed strong, obvious associations between features, such as total gold and minion kills, which are likely to exhibit multicollinearity and not be very useful for later analysis and prediction. Therefore, we've decided to shift our focus to exploring features based on individual roles rather than team features.

### Explore individual role data

We'll start by investigating the statistics of bot-lane players. The bot-lane role is particularly intriguing because it is often central to a team's strategy. In most games, bot-lane players choose ranged champions that deal damage primarily through regular attacks rather than skills. They typically start the game weak and need resources to become stronger in the late game. One common strategy is for the team to invest effort in allowing the bot-laner to gain resources (gold through kills, assists, and minion kills) and protect them during combat so they can deal maximal damage to opponent champions. Since the bot-lane role often deals substantial and consistent damage for the team during the late game, a stronger bot-laner generally improves the team's chances of winning. Therefore, to address our central question about team wins and losses, it's worthwhile to explore bot-lane data and the dynamic between bot-lane and other support roles.

#### Bivariate analyses

Questions of interest:

- Does the vision score (VS) from supports (and/or junglers, mid-laners) help bot-laners earn more resources and become stronger?
- Is there a relationship between bot-laners' creep score (CS) and damage to champions?
- In general, do bot-laners deal the most damage to enermy champions per gold spent?

#####  Bot-lane total creep score vs. Supportive vision score

In [13]:
# Retrieve individual role data
bot = get_position_stats(raw, 'bot', ['total cs', 'damagetochampions', 'kills'])
sup = get_position_stats(raw, 'sup', ['visionscore', 'vspm'])
jng = get_position_stats(raw, 'jng', ['visionscore', 'vspm'])
mid = get_position_stats(raw, 'mid', ['visionscore', 'vspm'])

combined = (bot.merge(sup, on=['gameid', 'result'])
            .merge(jng, on=['gameid', 'result'])
            .merge(mid, on=['gameid', 'result']))
combined['sup_jng_mid_visionscore'] = (combined['sup_visionscore']
                                       + combined['jng_visionscore']
                                       + combined['mid_visionscore'])
print('Correlation between `bot_toal cs` vs. `sup_jng_mid_visionscore`:')
combined['bot_total cs'].corr(combined['sup_jng_mid_visionscore'])

Correlation between `bot_toal cs` vs. `sup_jng_mid_visionscore`:


0.740155663469082

In [14]:
small_combined = combined.sample(1000)
fig = px.scatter(
    small_combined,
    x='sup_jng_mid_visionscore',
    y='bot_total cs',
    title="Bot-lane Total Creep Score vs. Supportive Vision Score",
    color_discrete_sequence=['#3288bd'],
    labels={
        'sup_jng_mid_visionscore': 'Support, Jungler, and Mid-Lane Vision Score',
        'bot_total cs': 'Bot-Lane Total Creep Score'
    }
)
fig.show()

There is a strong positive linear correlation between the bot-lane's creep score and the total vision scores of the support, mid-lane, and jungle combined. This suggests that better vision control is associated with a higher creep score for the bot-lane, whose role is to deal as much damage to opponents as possible in the late game.

Additionally, there is a noticeable difference in the trends for the bot-lane's total creep score when comparing scores below and above 100. One plausible explanation is that games where the bot-lane has a low creep score tend to end early. Consequently, the association between vision score and creep score in these cases may differ from games where the bot-lane has a higher creep. Since we are more interested in the impact of bot-laners in the late game and on game results, we will ignore the instances where the bot-lane creep score is less than 100.

In [15]:
# Ignore rows where `bot_total cs` is then less than 100
bot_cs = get_position_stats(raw, 'bot', ['total cs', 'damagetochampions', 'goldspent'])
bot_cs = bot_cs[bot_cs['bot_total cs'] > 100]
bot_cs.sample(4)

,gameid,result,bot_total cs,bot_damagetochampions,bot_goldspent
40652,ESPORTSTMNT01_2772766,False,185.0,11673.0,8005.0
9135,ESPORTSTMNT01_2697682,False,260.0,19030.0,14650.0
31683,ESPORTSTMNT03_2590119,False,196.0,4974.0,8225.0
125583,ESPORTSTMNT01_3085566,True,331.0,21469.0,15800.0


##### Gold spent by bot-lane vs. Bot-lane total creep score
Why does creep score so important to bot-laners (although it is also important to other roles too)? We can easily see a strong positive linear relationship between creep score and gold spent:

In [16]:
small_bot_cs = bot_cs.sample(1000)

fig = px.scatter(
    small_bot_cs,
    x='bot_total cs', y='bot_goldspent',
    color_discrete_sequence=['#3288bd'],
    title='Bot-lane Total Creep Score vs. Gold Spent by Bot-laner',
    labels={
        'bot_total cs': 'Bot-lane Total Creep Score',
        'bot_goldspent': 'Gold Spent by Bot-laner'
    }
)
fig.show()

##### Bot-lane's damage to champions vs. Gold spent by bot-laner

Creep score is the primary resource for a bot laner, especially in the early game when it is harder for them to earn gold through kills or destroying structures. Gold spent is the direct resource for any player to get stronger since they can buy items to increase their damage. Consequently, it is intuitive to see a positive trend between bot-laners' gold spent and their damage to enemy champions.

In [17]:
fig = px.scatter(
    small_bot_cs,
    x='bot_goldspent', y='bot_damagetochampions',
    color_discrete_sequence=['#3288bd'],
    title='Bot-lane\'s Damage to Champions vs. Gold Spent by Bot-laner',
    labels={
        'bot_goldspent': 'Gold Spent by Bot-laner',
        'bot_damagetochampions': 'Bot-lane\'s Damage to Champion'
    }
)
fig.show()

#### Damage to champions per gold spent by roles

So far, we have verified the positive relationships between bot-laners' resource gains and their damage to enemy champions. This also raises an interesting question: Does gold spending (the use of resource gain) in other roles similarly contribute to dealing damage to enemies? Let's compare the damage dealt to enemy champions by each role per gold they spent.

In [18]:
dtc_per_goldspent = raw[['position', 'damagetochampions', 'goldspent']]
dtc_per_goldspent['dtc_per_goldspent'] = (dtc_per_goldspent['damagetochampions']
                                          / dtc_per_goldspent['goldspent'])

fig = px.box(dtc_per_goldspent, x='position', y='dtc_per_goldspent', color='position',
             title='Damage to Champions per Gold Spent by Position',
             labels={
                 'gold_spent': 'Gold Spent',
                 'position': 'Roles',
                 'dtc_per_goldspent': 'Damage to Champions per Gold Spent'
             })
role_labels = {
    'top': 'Top-lane',
    'jng': 'Jungler',
    'mid': 'Mid-lane',
    'bot': 'Bot-lane',
    'sup': 'Support'
}
fig.update_xaxes(tickvals=list(role_labels.keys()), ticktext=list(role_labels.values()))
for trace in fig.data:
    trace.name = role_labels.get(trace.name, trace.name)
fig.show()

As we can see, mid-laners and bot-laners typically deal the most damage to champions by utilizing the resources they gain, primarily gold. Junglers and supports, on the other hand, focus more on supporting their teammates by facilitating resource gains and disrupting opponents, often sacrificing their own damage output. This difference in roles is reflected in the plot.

For top-laners, their damage to champions per gold spent is slightly lower than that of bot-laners and mid-laners. This could be due to the fact that top-laners are usually the strongest champions (both offensively and defensively) at the start of the game, although there are exceptions depending on specific champions. As the game progresses, bot-laners and mid-laners become stronger through acquiring more items from the resources they gain, leading to them dealing much more damage during late-game combat. Additionally, top-laners often transition into 'tankers,' who absorb damage from enemy champions. Let's see if our data reflects this pattern in League of Legends games by observing the trend of damage dealt by top-laners, mid-laners, and bot-laners as the game length increases.

#### Top, mid, bot-laners' damage to champions by game length

In [19]:
# Average game length by bins and group by roles
dtc_gl = (raw[raw['position']
              .isin(['top', 'mid', 'bot'])]
          [['position', 'gamelength', 'damagetochampions']])
dtc_gl['gamelength_bin'] = pd.cut(dtc_gl['gamelength'], bins=10)
dtc_gl = dtc_gl.groupby(['position', 'gamelength_bin']).mean().reset_index()

# Plot the data
fig = px.line(dtc_gl, x='gamelength', y='damagetochampions', color='position',
              title='Damage to Champions vs. Game Length by Position',
              labels={'gamelength': 'Game Length',
                      'damagetochampions': 'Damage to Champions'})

# Update the layout for better presentation
fig.update_layout(xaxis_title='Game Length (minutes)', yaxis_title='Damage to Champions')
fig.show()

It is observed that, on average, bot-laners relatively deal more damage to champions as the game progresses. While the data doesn't reflect our initial expectations regarding top-laners' early game damage, it does show that bot-laners and mid-laners are the roles that deal the most damage to champions in the late game. This aligns with the typical roles of these positions as the primary damage dealers or "carries" of a team. We will revisit the analysis of the carrying potential of bot-laners and mid-laners in the hypothesis testing section.

## Step 3: Assessment of Missingness

### Investigate missingness on columns of interest

So far, the columns of interest in the original dataset are:

In [20]:
columns = [
    'gameid', 'side', 'result', 'position', 'kills', 'deaths', 'assists',
    'damagetochampions', 'dpm', 'damagemitigatedperminute','visionscore',
    'vspm', 'totalgold', 'goldspent', 'minionkills', 'towers'
]

For the purpose of later analysis and prediction of game outcomes, we will also include the following nominal columns:

In [21]:
columns = columns + [
    'firstblood', 'firstdragon', 'dragons', 'firstbaron', 'barons',
    'firsttower', 'firsttothreetowers', 'inhibitors'
] + ['datacompleteness'] # Columns indicating whether the game data is complete or not

In [22]:
missing = raw[columns]
missing.head()

,gameid,side,result,position,...,firsttower,firsttothreetowers,inhibitors,datacompleteness
0,ESPORTSTMNT01_2690210,Blue,False,top,...,NaN,NaN,0.0,complete
1,ESPORTSTMNT01_2690210,Blue,False,jng,...,NaN,NaN,0.0,complete
2,ESPORTSTMNT01_2690210,Blue,False,mid,...,NaN,NaN,0.0,complete
3,ESPORTSTMNT01_2690210,Blue,False,bot,...,NaN,NaN,0.0,complete
4,ESPORTSTMNT01_2690210,Blue,False,sup,...,NaN,NaN,0.0,complete


In [23]:
# Show the number of missing values by columns
missing.isna().sum().to_dict()

{'gameid': 0,
 'side': 0,
 'result': 0,
 'position': 0,
 'kills': 0,
 'deaths': 0,
 'assists': 0,
 'damagetochampions': 12,
 'dpm': 12,
 'damagemitigatedperminute': 21828,
 'visionscore': 12,
 'vspm': 12,
 'totalgold': 0,
 'goldspent': 12,
 'minionkills': 3648,
 'towers': 124140,
 'firstblood': 18192,
 'firstdragon': 127778,
 'dragons': 124140,
 'firstbaron': 127778,
 'barons': 21900,
 'firsttower': 127778,
 'firsttothreetowers': 127778,
 'inhibitors': 21462,
 'datacompleteness': 0}

We can see that there are columns with identical number of missingness (e.g. `firstdragon`, `firstbaron`), we will analyze the missingness in these columns first since they might have the same missingness mechanism.

In [24]:
pd.options.display.max_rows = 50
pd.options.display.max_columns = 10

df = (missing.groupby('gameid')[['gameid', 'position', 'side', 'towers', 'firstdragon',
                                 'dragons', 'firstbaron', 'firsttower', 'firsttothreetowers']]
 .filter(lambda df: df['towers'].isna().sum() != 0))

(df[['gameid', 'position', 'side', 'firstdragon',
     'firstbaron', 'firsttower', 'firsttothreetowers']]
 .head(12))

,gameid,position,side,firstdragon,firstbaron,firsttower,firsttothreetowers
0,ESPORTSTMNT01_2690210,top,Blue,NaN,NaN,NaN,NaN
1,ESPORTSTMNT01_2690210,jng,Blue,NaN,NaN,NaN,NaN
2,ESPORTSTMNT01_2690210,mid,Blue,NaN,NaN,NaN,NaN
3,ESPORTSTMNT01_2690210,bot,Blue,NaN,NaN,NaN,NaN
4,ESPORTSTMNT01_2690210,sup,Blue,NaN,NaN,NaN,NaN
5,ESPORTSTMNT01_2690210,top,Red,NaN,NaN,NaN,NaN
6,ESPORTSTMNT01_2690210,jng,Red,NaN,NaN,NaN,NaN
7,ESPORTSTMNT01_2690210,mid,Red,NaN,NaN,NaN,NaN
8,ESPORTSTMNT01_2690210,bot,Red,NaN,NaN,NaN,NaN
9,ESPORTSTMNT01_2690210,sup,Red,NaN,NaN,NaN,NaN


It appears that if we focus solely on rows where the `position` column does not contain the value `team`, the missingness in these `first...` columns (i.e. `firstdragon`, `firstbaron`, etc.) is by design (MD). This is because we can infer from the `position` column whether there are missing values in these columns.

However, when the `position` column value is `team`, we cannot determine if there is a missing value in these `first...` columns. Therefore, we need to carefully analyze these rows by separating them from the full dataset, similar to how we analyzed game measurements in the previous section.

### Addressing Missingness in `first...` Columns in Game Rows

For the analysis of missingness in the `first...` columns only for rows where `position` is `team`, we will treat the DataFrame containing only these team rows as a separate dataset. Although the `first...` columns in the `raw` DataFrame contain a mix of individual role data and team data, they exhibit different missingness mechanisms when considered separately (as we have already verified the missingness mechanism of `first...` columns for individual role data as MD). This section will focus on analyzing the missingness mechanism for `first...` columns of team data alone.

In [25]:
# Filter game rows that has a NaN value
miss1 = (get_game_rows(df)
      .groupby('gameid')
      .filter(lambda df: df.isna().sum().sum() != 0))
print('Number of NaN values in each column of interests:')
miss1.isna().sum()

Number of NaN values in each column of interests:


gameid                   0
position                 0
side                     0
towers                   0
firstdragon           3638
dragons                  0
firstbaron            3638
firsttower            3638
firsttothreetowers    3638
dtype: int64

In [26]:
pd.options.display.max_columns = 8

miss1_games = miss1['gameid'].unique()
print('Number of games with \'first...\' missingness:', len(miss1_games))
full_miss1 = get_game_rows(raw[raw['gameid'].isin(miss1_games)])
full_miss1.sample(10)

Number of games with 'first...' missingness: 1819


,gameid,datacompleteness,url,league,...,deathsat15,opp_killsat15,opp_assistsat15,opp_deathsat15
107351,9194-9194_game_1,partial,https://lpl.qq.com/es/stats.shtml?bmid=9194,LDL,...,NaN,NaN,NaN,NaN
42911,8785-8785_game_1,partial,https://lpl.qq.com/es/stats.shtml?bmid=8785,LPL,...,NaN,NaN,NaN,NaN
81286,8965-8965_game_3,partial,https://lpl.qq.com/es/stats.shtml?bmid=8965,LPL,...,NaN,NaN,NaN,NaN
118534,9267-9267_game_1,partial,https://lpl.qq.com/es/stats.shtml?bmid=9267,LDL,...,NaN,NaN,NaN,NaN
82582,8968-8968_game_2,partial,https://lpl.qq.com/es/stats.shtml?bmid=8968,LPL,...,NaN,NaN,NaN,NaN
78322,8955-8955_game_2,partial,https://lpl.qq.com/es/stats.shtml?bmid=8955,LPL,...,NaN,NaN,NaN,NaN
17879,8479-8479_game_1,partial,https://lpl.qq.com/es/stats.shtml?bmid=8479,LDL,...,NaN,NaN,NaN,NaN
103283,9170-9170_game_1,partial,https://lpl.qq.com/es/stats.shtml?bmid=9170,LDL,...,NaN,NaN,NaN,NaN
120827,9339-9339_game_3,partial,https://lpl.qq.com/es/stats.shtml?bmid=9339,LPL,...,NaN,NaN,NaN,NaN
56555,8815-8815_game_2,partial,https://lpl.qq.com/es/stats.shtml?bmid=8815,LPL,...,NaN,NaN,NaN,NaN


Since the number of games with missing values (1819 games, which corresponds to 1819 * 2 = 3638 rows in the DataFrame) matches the number of missing values in each of the `first...` columns (3638 values), the missingness in these columns occurs in the same instances. Therefore, we can analyze and draw conclusions about them collectively.

It appears that game rows with missing `first...` columns are related to the values in the `datacompleteness` column.

In [27]:
full_miss1['datacompleteness'].value_counts()

partial    3638
Name: datacompleteness, dtype: int64

It is apparent that all the missing values occur in rows where `datacompleteness` is partial. However, this alone does not indicate a Missing by Design (MD) mechanism, as there are partial completeness rows that do not have missing values in these `first...` columns. Nevertheless, this observation prompts us to conduct a permutation test on the categorical distribution of the `datacompleteness` column for missing and non-missing values in the `first...` columns. The outcome of the test could suggest a Missing at Random (MAR) mechanism for the missingness.

### Permutation test for verifying MAR of `first...` columns

#### Column on which `first...`'s missingness depends

In [28]:
raw['datacompleteness'].value_counts()

complete    126612
partial      21828
ignore         528
Name: datacompleteness, dtype: int64

In [29]:
# Preprocess for permutation testing

# Create DataFrame for testing where each game has only one row
def process_perm_test_df(raw, miss1_games, test_col):
    raw = get_game_rows(raw)
    raw = raw[raw['side'] == 'Blue']
    
    all_games = raw['gameid'].unique()
    non_miss1_games = [game for game in all_games if game not in miss1_games]

    miss1_df = raw[raw['gameid'].isin(miss1_games)][[test_col]]
    miss1_df['is_missing'] = [True] * miss1_df.shape[0]

    non_miss1_df = raw[raw['gameid'].isin(non_miss1_games)][[test_col]]
    non_miss1_df['is_missing'] = [False] * non_miss1_df.shape[0]

    return pd.concat([miss1_df, non_miss1_df], ignore_index=True)

miss1_perm_df = process_perm_test_df(raw, miss1_games, 'datacompleteness')
miss1_perm_df.sample(4)

,datacompleteness,is_missing
8800,complete,False
12025,complete,False
1032,partial,True
9612,complete,False


In [30]:
# Perform permutation testing
def permutation_test_tvd(df, cat_col, missingness_col, n_rep=1000):
    shuffled = df.copy()
    def compute_tvd(df):
        pivoted = (
            df.pivot_table(index=cat_col,
                           columns=missingness_col,
                           aggfunc='size')
        ).fillna(0)
        pivoted = pivoted / pivoted.sum()
        return pivoted.diff(axis=1).iloc[:, -1].abs().sum() / 2

    tvds = []
    for _ in range(n_rep):
        shuffled[cat_col] = np.random.permutation(shuffled[cat_col])
        tvd = compute_tvd(shuffled)
        tvds.append(tvd)

    observed_tvd = compute_tvd(df)
    p_value = (tvds > observed_tvd).mean()
    
    return p_value, tvds, observed_tvd

p_value, tvds, observed_tvd = permutation_test_tvd(miss1_perm_df,
                                                   'datacompleteness',
                                                   'is_missing')    
print(f'p-value of the test: {p_value}')

p-value of the test: 0.0


In [31]:
# Plot the null distribution
fig = px.histogram(pd.DataFrame(tvds), x=0, nbins=40, histnorm='probability', 
                   title='Empirical Distribution of the TVD')
fig.update_traces(marker_color='#3288bd', opacity=0.9)
fig.add_annotation(
    text=f'<span style="color:steelblue">Observed TVD = {round(observed_tvd, 2)}</span>',
    x=0.025, showarrow=False, y=0.08)
fig.update_layout(xaxis_title='Total Variation Distance',
                  yaxis_title='Probability')
fig.show()

As p-value of the test is extremely small, it's evident that the missing rows in the `first...` columns have a distribution distinct from that of the non-missing rows, as indicated by the values in `datacompleteness`. Therefore, since the missingness in `first... columns` depends on `datacompleteness`, their missing mechanism is Missing at Random (MAR).

#### Column Independent of first... Missingness

Furthermore, we will conduct another permutation test on the `result` column in relation to the missingness of the `first...` columns.

In [32]:
# Test on a column that is unlike to have dependency with the missingness
test_on_result = process_perm_test_df(raw, miss1_games, 'result')
p_value, tvds, observed_tvd = permutation_test_tvd(test_on_result, 'result', 'is_missing')
print(f'p-value of the test: {p_value}')

# Plot the null distribution
fig = px.histogram(pd.DataFrame(tvds), x=0, nbins=40, histnorm='probability', 
                   title='Empirical Distribution of the TVD')
fig.add_annotation(text=f'<span style="color:steelblue">Observed TVD = {round(observed_tvd, 2)}</span>',
                   x=0.025, showarrow=False, y=0.08)
fig.update_traces(marker_color='#3288bd', opacity=0.9)
fig.add_vline(x=observed_tvd, line_width=2, line_dash="dash", line_color="red")
fig.update_layout(xaxis_title='Total Variation Distance',
                  yaxis_title='Probability')
fig.show()

p-value of the test: 0.417


Since the large p-value indicates an insignificant difference between the two groups, we failed to reject the null hypothesis. Therefore, we conclude that the missingness in the `first...` columns does not depend on the `result` column.

## Step 4: Hypothesis Testing

Through exploring the data, we have found that mid-laners and bot-laners generally do the most damage to champions per gold spent. This observation inspired a key question: Which role, mid-laners or bot-laners, contributes more damage overall when carrying their team? Let's explore this question further.

To answer this, we need to define what it means to 'carry' and determine the appropriate metric for measuring how much a role carries their team.

**Defining 'Carry'**: In this context, a 'carry' is defined as the role that deals the most damage on their team in a game. This damage includes all forms: damage to champions, minions/monsters, and structures, as each type of damage contributes significantly to the team's victory.

We focus on damage rather than kills because damage is a more direct measure of contribution to a team's success, whereas kills can be influenced by the specific champions being played (some champions can secure kills more easily than others).

**Metric**: We choose to use the amount of damage dealt per minute as our metric. Specifically, we look at cases where mid-laners or bot-laners deal the most damage for their team to measure how much damage they have dealt when they carry their team. We exclude data points where neither the bot-laner nor mid-laner deals the most overall damage, as the carrying role in these cases would be another position. These exclusions help avoid complications from unconventional picks, game strategies, or unusually short games.

In [33]:
# Retrieve a DataFrame for testing
testing = get_player_rows(raw)[['gameid', 'side', 'position', 'dpm']]
print(testing.isna().sum().rename('Missingness'))
print('The DataFrame\'s shape:', testing.shape)
testing.head()

gameid       0
side         0
position     0
dpm         10
Name: Missingness, dtype: int64
The DataFrame's shape: (124140, 4)


,gameid,side,position,dpm
0,ESPORTSTMNT01_2690210,Blue,top,552.29
1,ESPORTSTMNT01_2690210,Blue,jng,412.08
2,ESPORTSTMNT01_2690210,Blue,mid,499.40
3,ESPORTSTMNT01_2690210,Blue,bot,389.00
4,ESPORTSTMNT01_2690210,Blue,sup,128.30


The missing values of `dpm` belong to one game (10 rows for 10 players). These missing rows constitute a very small portion of the dataset (representing only 2 data points out of 124,140 / 5 = 24,828 data points). Therefore, we will remove them from our analysis dataset without significantly impacting the test results.

In [34]:
testing = testing.dropna()

# Transform the data to access the sample
testing['position_dpm'] = list(zip(testing['position'], testing['dpm']))

def mid_bot_carry(pos_dpm):
    max_pos_dmp = max(pos_dpm, key=lambda x: x[1])
    if not max_pos_dmp[0] in ['mid', 'bot']:
        return np.nan
    return max_pos_dmp

roles, dpm = zip(*testing
                 .groupby(['gameid', 'side'])['position_dpm']
                 .aggregate(mid_bot_carry)
                 .dropna()
                 .reset_index(drop=True))
sample = pd.DataFrame({'carry_role': roles, 'dpm': dpm})
sample.sample(5)

,carry_role,dpm
9056,bot,518.18
2304,bot,549.52
17117,bot,785.51
184,bot,765.88
3306,mid,417.75


Let's first visualize the observed distributions of `dpm` from mid-laners and bot-laners:

In [35]:
# Create the histogram
fig = px.histogram(sample, x='dpm', color='carry_role', histnorm='probability',
                   marginal='box', barmode='overlay', opacity=0.7,
                   color_discrete_sequence=['#3288bd', '#d53e4f'],
                   title='Distribution of Damage per Minute grouped by Carry Role')

# Update layout for better visualization
fig.update_layout(
    legend_title='Carry role',
    xaxis_title='Damage per Minute',
    yaxis_title='Probability',
    showlegend=True
)
fig.show()

It appears that the average damage dealt by carry bot-laners is slightly greater than the average damage dealt by carry mid-laners. Let's perform a permutation test to formally determine if the average damage dealt by carry bot-laners is significantly greater.

*Note*: Since we only have two samples of damage distributions from carry bot-laners and carry mid-laners without any well-defined population distribution models, we choose to use a permutation test as the suitable method for this case.

Let's establish our hypotheses:
- **Null hypothesis**: The damage dealt by carry bot-laners and carry mid-laners comes from the same distribution.
- **Alternative hypothesis**: The damage dealt by carry bot-laners comes from a distribution that is greater than that of carry mid-laners.
- **Test statistic**: The difference in group means of damage per minute (bot-laners' mean `dpm` minus mid-laners' mean `dpm`).

In [36]:
n_rep = 1000
shuffled = sample.copy()

def compute_diff_group_means(df):
    grouped = df.groupby('carry_role').mean()
    return grouped.loc['bot', 'dpm'] - grouped.loc['mid', 'dpm']

# Simulate
diffs = []
for _ in range(n_rep):
    shuffled['carry_role'] = np.random.permutation(shuffled['carry_role'])
    diff = compute_diff_group_means(shuffled)
    diffs.append(diff)

# Compute observed statistic
observed_diff = compute_diff_group_means(sample)

# Report
print('p-value of the test:', (diffs >= observed_diff).mean())

fig = px.histogram(pd.DataFrame(diffs), x=0, nbins=40, histnorm='probability', 
                   title='Empirical Distribution of the Differences in Group Means')
fig.update_traces(marker_color='#3288bd', opacity=0.9)
fig.add_vline(x=observed_diff, line_width=2, line_dash="dash", line_color="red")
fig.add_annotation(
    text=f'<span style="color:steelblue">Observed Difference = {round(observed_diff, 2)}</span>',
    x=13, showarrow=False, y=0.1)
fig.update_layout(xaxis_title='Difference in Group Mean',
                  yaxis_title='Probability')
fig.show()

p-value of the test: 0.0


**Permutation Test Conclusion**: The test provided strong evidence that the mean damage dealt by carry bot-laners is significantly greater than that dealt by carry mid-laners. Therefore, we are more confident to believe that, in general, bot-laners deal more damage when carrying their team compared to mid-laners.

## Step 5: Framing a Prediction Problem

As previously mentioned, our goal is to predict whether a team will win a game. We will develop a classifier that categorizes a team as 'win' or 'lose' based on their input features.

Importantly, our classifier aims to predict the outcome before the game's conclusion. Therefore, we will avoid using any features that can only be collected at the end of the game, such as total gold or the number of turrets destroyed. Although this means we cannot use some valuable features like total gold and damage per minute, which we identified as having strong patterns between winning and losing in earlier analyses, we can leverage in-game features that might correlate strongly with these end-game statistics. For example, gold at the 15-minute mark might have a strong linear relationship with total gold by the end of the game. We will investigate these in-game features to establish a baseline model in the next section.

Additionally, there are many potential nominal features that could be engineered to enhance our classifier, such as the champions picked or whether a team secured the first blood or first baron. We will explore these features in subsequent sections.

## Step 6: Baseline Model

### Decide features

Since `totalgold` shows clear patterns between winning and losing, we will consider a team's `goldat10` and/or `goldat15` as the first feature(s) for our classifier. Let's first look at the correlations between these features and `total gold`:

In [37]:
print('Correlation scores between \'totalgold\' and other features')
raw.corr()['totalgold'].abs().sort_values(ascending=False).head(10)

Correlation scores between 'totalgold' and other features


totalgold     1.00
goldspent     1.00
earnedgold    0.99
earned gpm    0.96
goldat15      0.96
goldat10      0.96
xpat15        0.95
xpat10        0.95
csat15        0.95
csat10        0.94
Name: totalgold, dtype: float64

We observe that `goldat10` and `goldat15` exhibit the strongest correlations with totalgold among in-game features. To establish a baseline model, we'll leverage these two quantitatice features as predictors in a `DecisionTreeClassifier`.

Our choice of `DecisionTreeClassifier` is informed by previous data exploration, which revealed distinct patterns in `totalgold` distribution across two classes of `result` (win and lose).

Additionally, we include `firstbaron` as a nominal feature for the classifier. Our prior analysis has showed that securing the first Baron significantly increases the likelihood of winning the game, making it a valuable feature for our baseline model.

To ensure generalization and hyperparameter tuning, we'll split the data into training and test sets and employ k-fold cross-validation. Specifically, we'll utilize `GridSearchCV` from `sklearn` to efficiently optimize the `max_depth` hyperparameter. Our evaluation metric will be accuracy score, which we'll use consistently throughout this analysis to assess the performance of subsequent models relative to this baseline.

### Process the data and train the model

In [38]:
from sklearn.model_selection import train_test_split

# Prepare the data
predict_df = (get_game_rows(raw)
              [['gameid', 'side', 'result', 'goldat10', 'goldat15', 'firstbaron']]
              .dropna()
              .reset_index(drop=True))

# Split the data
X_train, X_test, y_train, y_test = train_test_split(predict_df[['goldat10', 'goldat15']],
                                                    predict_df['result'],
                                                    random_state=1)
print(
      f'X_train shape: {X_train.shape}\n'
    + f'y_train shape: {y_train.shape}\n'
    + f'X_test shape:  {X_test.shape}\n'
    + f'y_test shape:  {y_test.shape}\n'
)

X_train shape: (15892, 2)
y_train shape: (15892,)
X_test shape:  (5298, 2)
y_test shape:  (5298,)



In [39]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

hyperparams = {'max_depth': [2, 4, 6]}
simplest_dt = GridSearchCV(DecisionTreeClassifier(), param_grid=hyperparams, cv=5)
simplest_dt.fit(X_train, y_train)

# Predict and compute accuracy on the train set
train_accuracy = accuracy_score(y_train, simplest_dt.predict(X_train))
print(f'Training set accuracy: {train_accuracy}')

# Predict and compute accuracy on the test set
test_accuracy = accuracy_score(y_test, simplest_dt.predict(X_test))
print(f'Test set accuracy: {test_accuracy}')

Training set accuracy: 0.6912912157060156
Test set accuracy: 0.6864854662136656


### Evaluate the baseline model, next step

The baseline model's training accuracy indicates its suitability for prediction, as it significantly exceeds 0.5, outperforming random guessing in binary classification. The test accuracy, which closely mirrors the training accuracy, further suggests that the model generalizes well. As a result, we will use this model's design as a baseline for future improvements and set the test accuracy as the benchmark for evaluating subsequent enhancements.

## Step 7: Final Model

### Prepare

To enhance our classifier, we need to engineer additional useful features. We will begin by incorporating more in-game features into our dataset. Additionally, to ensure the generalization of the final model, we will design and refine the model exclusively using the training set.

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Define a general model template for training and evaluating
# the train set on the go

def update_features(predict_df):
    """
    Update new features to the existing training set and test set
    
    Note: this doesn't change the instances existed in the
    previous training-test split since we are using the same
    random_state for spliting
    """
    X_train, X_test, y_train, y_test = train_test_split(
        predict_df.drop(['result', 'gameid', 'side'], axis=1),
        predict_df['result'],
        random_state=1)
    return X_train, X_test, y_train, y_test

def train(X, y, onehot_cols=None, hyperparams=None):
    preproc = ColumnTransformer(
        transformers=[
            ('one-hot', OneHotEncoder(handle_unknown='ignore'), onehot_cols)
        ],
        remainder='passthrough'
    )
    pl = Pipeline([
        ('preprocess', preproc),
        ('dec-tree', DecisionTreeClassifier())
    ])
    model = GridSearchCV(pl, param_grid=hyperparams, cv=5)
    
    return model.fit(X, y), model.best_params_

def get_train_accuracy(model, X, y):
    return accuracy_score(y, model.predict(X))

### Add pick and patch features, one-hot encode

The ban/pick stage is a crucial part of a game that can significantly impact a team's strategy and winning chances, especially in professional play. Professional players often specialize in one or two roles and have a set of champions they have mastered. Additionally, certain champions may be stronger or more suitable for the trending tactics of the current game version. Thus, it is highly advantageous if a professional player can pick one of their skilled champions, particularly if it is one of the season's "hot" picks.

Given this context, knowing a team's picks is undoubtedly a valuable predictor for game outcomes. Therefore, we will include picking and version information as predictive features for our classifier to learn these complexities.

Since both picked champions and patches (small version updates) are nominal, we plan to employ one-hot encoding for feature engineering. We recognize that some champion names may appear in the test set but not in the training set. For simplicity, we will set the `handle_unknown` parameter in `OneHotEncoder` to `'ignore'`.

Furthermore, we acknowledge that the model may not generalize well to predict results for games in other years, as game versions and "hot" picks change constantly. However, this approach can still provide a framework for building predictive models with other datasets.

In [41]:
# Include champions to the dataset
lane_champion = (get_player_rows(raw).pivot(index=['gameid', 'side'],
                                            columns='position',
                                            values='champion')
                 .reset_index())

predict_df = predict_df.merge(lane_champion, on=['gameid', 'side'], how='left')
predict_df = (predict_df.merge(get_game_rows(raw)
                               [['gameid', 'side', 'patch']], on=['gameid', 'side'], how='left'))
print('New added features:')
predict_df.iloc[:3, -6:]

New added features:


,bot,jng,mid,sup,top,patch
0,Samira,Xin Zhao,LeBlanc,Leona,Renekton,12.01
1,Jinx,Viego,Viktor,Alistar,Gragas,12.01
2,Jhin,Lee Sin,Orianna,Rakan,Gragas,12.01


In [42]:
# Train and evaluate the model on new champions and patch one-hot-encoding features
X_train, X_test, y_train, y_test = update_features(predict_df) # This doesn't change the original split
model, best_hyperparams = train(X_train, y_train, onehot_cols=['bot', 'jng', 'mid', 'sup', 'top', 'patch'],
                                hyperparams={'dec-tree__max_depth': [2, 4, 6],
                                             'dec-tree__criterion': ['gini', 'entropy']})
print('Best hyperparameters:', best_hyperparams)
print('Model\'s train accuracy:', get_train_accuracy(model, X_train, y_train))

Best hyperparameters: {'dec-tree__criterion': 'entropy', 'dec-tree__max_depth': 4}
Model's train accuracy: 0.8471558016612132


Since the training accuracy shows a significant improvement from the baseline model, we will incorporate these new features into our final model.

### Add `first...` features

We are also incorporating "first-take-down" features (e.g., first blood, first dragon) into our model. We believe these features are useful as they provide information about a team's early game performance. While these features may not be as strong indicators as firstbaron (a mid-late game feature) for a team's winning chance, they still offer valuable insights into the dynamic relationship between early and late game performance that the model might be able to capture.

In [43]:
predict_df = (predict_df
              .merge(get_game_rows(raw)
                     [['gameid', 'side', 'firstblood', 'firstdragon', 'firstherald',
                       'firsttower', 'firstmidtower', 'firsttothreetowers']],
                     on=['gameid', 'side'], how='left'))
print('New added features:')
predict_df.iloc[:3, -6:]

New added features:


,firstblood,firstdragon,firstherald,firsttower,firstmidtower,firsttothreetowers
0,True,False,True,True,True,True
1,False,True,False,False,False,False
2,False,False,True,False,False,False


Before moving on, we need to resolve a minor missingness in `firstmidtower` column.

In [44]:
# Resolve missingness by probabilistic imputation
ms = predict_df.isna().sum().sort_values(ascending=False)
print('Number of missing values:')
print(ms.iloc[:3])
p_fills = np.random.choice([True, False], ms.iloc[0])
predict_df['firstmidtower'][predict_df['firstmidtower'].isna()] = p_fills
print('Missing values after imputation:', predict_df.isna().sum().sum())

Number of missing values:
firstmidtower    2
gameid           0
side             0
dtype: int64
Missing values after imputation: 0


In [45]:
# Train and evaluate the model on first-take-down features
X_train, X_test, y_train, y_test = update_features(predict_df) # This doesn't change the original split
model, best_hyperparams = train(X_train, y_train, onehot_cols=['bot', 'jng', 'mid', 'sup', 'top', 'patch'],
                                hyperparams={'dec-tree__max_depth': [3, 5, 7],
                                             'dec-tree__criterion': ['gini', 'entropy']})
print('Best hyperparameters:', best_hyperparams)
print('Model\'s train accuracy:', get_train_accuracy(model, X_train, y_train))

Best hyperparameters: {'dec-tree__criterion': 'entropy', 'dec-tree__max_depth': 5}
Model's train accuracy: 0.852441479989932


Adding these early game features improves the training accuracy by a noticeable amount. We also observe that the optimal maximum depth of our Decision Tree classifier increases by 1, indicating increased model complexity. This is unlikely to cause overfitting since the optimal maximum depth stops at 5 rather than continuing to increase.


### Decide a candidate for the final model

Since the improved model's training accuracy has increased significantly compared to the baseline model's test accuracy, the current model has learned to better capture the data's complexity and reduce bias. Therefore, we have decided to evaluate it on the test set to see if it can generalize to unseen data.

In [46]:
# Compute accuracy on the test set
test_accuracy = accuracy_score(y_test, model.predict(X_test))
print(f'Test set accuracy: {test_accuracy}')

Test set accuracy: 0.8361645904114761


Since the test accuracy is close to the training accuracy and reflects an improvement over the baseline model, we consider the current model a strong candidate for the final model.

### Experiment with `RandomForestClassifier`

To further improve the model, we decide to fit all the engineered features to a `RandomForestClassifier` to see if the accuracy increases. Our motivation is to determine if introducing randomness to our features, while using the same learning principles, can produce a better classifier. The idea is that ensemble learning from random subsets of the training data and engineered features might help in generalizing the model while also potentially improving the complexity of the learning to reduce the model's bias.

In [47]:
from sklearn.ensemble import RandomForestClassifier

# Improve the model using RandomForestClassifier
def train_randomforest(X, y, onehot_cols=None, hyperparams=None):
    preproc = ColumnTransformer(
        transformers=[
            ('one-hot', OneHotEncoder(handle_unknown='ignore'), onehot_cols)
        ],
        remainder='passthrough'
    )
    pl = Pipeline([
        ('preprocess', preproc),
        ('rdfr-clr', RandomForestClassifier())
    ])
    model = GridSearchCV(pl, param_grid=hyperparams, cv=5)
    return model.fit(X, y), model.best_params_

In [48]:
# Train and evaluate the Random Forest classifier
rdfr_model, best_rdfr_hyperparams = train_randomforest(
    X_train, y_train,
    onehot_cols=['bot', 'jng', 'mid', 'sup', 'top', 'patch'],
    hyperparams={
        'rdfr-clr__n_estimators': [100, 150, 200],
        'rdfr-clr__max_depth': [5, 10, 15],
        'rdfr-clr__criterion': ['gini', 'entropy']
    })
print('Best hyperparameters:', best_rdfr_hyperparams)
print('Model\'s train accuracy:', get_train_accuracy(rdfr_model, X_train, y_train))

Best hyperparameters: {'rdfr-clr__criterion': 'gini', 'rdfr-clr__max_depth': 15, 'rdfr-clr__n_estimators': 150}
Model's train accuracy: 0.8731437201107476


In [49]:
# Compute accuracy on the test set
test_accuracy = accuracy_score(y_test, rdfr_model.predict(X_test))
print(f'Test set accuracy: {test_accuracy}')

Test set accuracy: 0.8225745564363911


It is observed that although the training accuracy for this model is greater than that of our current best model, the test accuracy does not show any improvement. This contrasts with our goal of using the `RandomForestClassifier` to enhance generalization. Since employing the `RandomForestClassifier` not only fails to improve the classifier's performance but also increases the computational cost of training, we have decided to discard this model and continue using our final `DecisionTreeClassifier`.

### Finalize the final model

Since we are staying with our best model using the `DecisionTreeClassifier`, we will finalize it by fitting the entire dataset to the model and computing the final accuracy.

In [50]:
X = predict_df.drop(['result', 'gameid', 'side'], axis=1)
y = predict_df['result']
final_model, _ = train(X, y, 
                    onehot_cols=['bot', 'jng', 'mid', 'sup', 'top', 'patch'],
                    hyperparams={'dec-tree__max_depth': [5],
                                 'dec-tree__criterion': ['entropy']})
print('Final model\'s accuracy:', accuracy_score(y, final_model.predict(X)))

Final model's accuracy: 0.8471448796602171


## Step 8: Fairness Analysis

### Identify groups

Upon fitting the entire dataset to our final model, a question arises about the fairness of our classifier: *Does our classifier perform better for teams/games in Tier 1 professional leagues than it does for teams/games in lower leagues?* This question stems from our intuition that our assumptions about game tactics and team behaviors when we built our model might be more accurate for professional players/teams in Tier 1 leagues than for those in lower leagues. It is possible that the tactics and players in Tier 1 leagues are more consistent than those in lower leagues. Although it is uncertain whether our model's performance might favor the Tier 1 group, it is reasonable to conduct a fairness analysis on these two groups.

To address this question, we will divide our dataset into two groups: one containing data from Tier 1 leagues and the other containing data from lower leagues. We will then use our final trained model to classify the results (win/lose) with the labeled groups and compute their accuracy scores, using the same metric we used to build the final model. Our goal is to conduct a statistical analysis to determine if our model performs fairly across both groups.

In [51]:
# Label the dataset and get model prediction

tier1_leagues = ['LCK', 'LPL', 'LEC', 'LCS', 'PCS', 'VCS', 'CBLOL', 'LLA']
# Get league info
fairness = predict_df.merge(get_game_rows(raw)[['gameid', 'side', 'league']],
                            on=['gameid', 'side'],
                            how='left')
# Label league groups
fairness['in_tier_one'] = fairness['league'].isin(tier1_leagues)
# Predict
fairness['predicted_result'] = (
    final_model
    .predict(fairness.drop(['result', 'gameid', 'side', 'league', 'in_tier_one'],
                           axis=1)))
# Retrive necessary data for analysis
fairness = fairness[['in_tier_one', 'result', 'predicted_result']]
fairness.sample(5)

,in_tier_one,result,predicted_result
10715,False,True,True
2951,False,True,True
17728,True,True,True
7605,True,True,True
4779,True,False,False


### Perform permutation test

As we already have the fairness-grouped data as well as the actual, predicted values, we are going to perform a permutation test using the **absolute difference in group accuracy scores** as the test statistic to see if our model's performance biases one group over the other. Here is the test's hypotheses:

- Null hypothesis: Our model is fair. Its accuracy for Tier 1 leagues and lower leagues are roughly the same, and any differences are due to random chance.
- Alternative Hypothesis: Our model is unfair. Its accuracy for Tier 1 leagues is different from its accuracy for lower leagues.

In [52]:
# Perform permutation test
n_rep = 1000
shuffled = fairness.copy()

def compute_abs_diff_acc(df):
    tier1 = df[df['in_tier_one']]
    non_tier1 = df[~df['in_tier_one']]
    diff = (accuracy_score(tier1['result'], tier1['predicted_result'])
           - accuracy_score(non_tier1['result'], non_tier1['predicted_result']))
    return abs(diff)

abs_diffs = []
for _ in range(n_rep):
    shuffled['in_tier_one'] = np.random.permutation(shuffled['in_tier_one'])
    abs_diff = compute_abs_diff_acc(shuffled)
    abs_diffs.append(abs_diff)

observed_abs_diff = compute_abs_diff_acc(fairness)
p_value = (abs_diffs >= observed_abs_diff).mean()
print('p-value of the test:', p_value)

p-value of the test: 0.96


In [53]:
# Report
fig = go.Figure()
fig.add_trace(go.Histogram(x=abs_diffs, nbinsx=40, histnorm='probability'))
fig.update_traces(marker_color='#3288bd', opacity=0.9)
fig.add_vline(x=observed_abs_diff, line_width=2, line_dash="dash", line_color="red")
fig.add_annotation(
    text=f'<span style="color:steelblue">Observed Score Difference = {round(observed_abs_diff, 4)}</span>',
    x=0.01, y=0.15, showarrow=False)
fig.update_layout(title='Empirical Distribution of the Absolute Accuracy Score Differences',
                  xaxis_title='Absolute Accuracy Score Difference',
                  yaxis_title='Probability')
fig.show()

### Fairness analysis conclusion

As we failed to reject the null hypothesis with a p-value of 0.9, the permutation test provided a strong evident that the difference between the accuracy scores of our classifier for Tier 1 leagues and lower league groups is not significant. Thus we are confident that our classifier's performance is fair for both Tier 1 leagues and lower leagues groups.